# 0.1b — Same exercise on the OLMo 3 base model

A copy of 0.1 with only the model name changed, to check whether `allenai/Olmo-3-1025-7B` shows the
instruction-tuned behaviour we saw in Qwen2.5-7B base (emitting `eos` after a single answer, markdown
lists, "As an AI language model"). OLMo's pretraining data is public, so this is the candidate for a
cleaner base/instruct comparison. Original notes from 0.1 follow.

**Goal.** Load `Qwen/Qwen2.5-7B` (the *pretrained* model, no post-training), feed it a plain-text
prompt with no chat template, and watch what it does. Two things to observe:

1. A base model is a text continuer, not an assistant. Given `User: ...\nAssistant:` it will write an
   answer, and then keep going: it will happily invent the next `User:` turn, and the one after that.
2. So we need a **stopping criterion**. We implement one by hand to see how generation actually
   works, then note the built-in equivalent.

Everything here is on purpose spelled out at the level of token IDs. Later notebooks build on this.

In [1]:
import os, time, json, textwrap
# HF env vars must be set BEFORE transformers is imported (read at import time); env.sh / the persona-ml kernel also set them.
os.environ.setdefault("HF_HOME", "/global/cfs/cdirs/m2612/ozamram/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

# --- Reproducibility & bookkeeping -----------------------------------------
# Every notebook saves its config next to its outputs so a result can always be traced
# back to exactly what produced it.
CONFIG = {
    "model": "allenai/Olmo-3-1025-7B",
    "dtype": "bfloat16",
    "seed": 0,
    "max_new_tokens": 150,
    # Where the base model would start hallucinating the next turn. The README suggests "\nUser:",
    # but in a first run the model wrote " User: ... Assistant: ..." all on one line, so match
    # "User:" regardless of what precedes it. (False positives, an answer that contains "User:",
    # are rare enough to ignore for now.)
    "stop_strings": ["User:"],
}
torch.manual_seed(CONFIG["seed"])

REPO = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().resolve().parent
RESULTS = REPO / "results" / "phase0"
RESULTS.mkdir(parents=True, exist_ok=True)

# Model weights live on CFS, not $HOME. `source env.sh` sets these, and so does the `persona-ml`
# Jupyter kernel; if you're on a different kernel, fall back to the same values here.
print("HF_HOME =", os.environ["HF_HOME"])
print("GPU:", torch.cuda.get_device_name(0))

HF_HOME = /global/cfs/cdirs/m2612/ozamram/hf_cache
GPU: NVIDIA A100-SXM4-80GB


## Load tokenizer and model

- `dtype=torch.bfloat16`: 7.6B params × 2 bytes ≈ 15 GB. fp32 would be 30 GB and no faster on an A100.
- `device_map="cuda"`: put the whole model on GPU 0. (`device_map="auto"` would shard across GPUs / CPU
  if it didn't fit; we don't need that.)
- The first load reads 15 GB from CFS; expect ~1 minute. Later loads are page-cache warm and faster.

In [2]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], dtype=torch.bfloat16, device_map="cuda")
model.eval()   # disables dropout etc. (no-op for inference here, but good hygiene)
if tokenizer.pad_token_id is None:      # OLMo's base tokenizer may not define a pad token; generate wants one
    tokenizer.pad_token = tokenizer.eos_token
print(f"loaded in {time.time()-t0:.0f}s")
print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB")

# Special tokens the *base* tokenizer knows about. Note eos == pad == <|endoftext|>.
# The instruct model adds <|im_start|>/<|im_end|> on top of this (notebook 0.2).
print("eos:", repr(tokenizer.eos_token), "| pad:", repr(tokenizer.pad_token), "| bos:", repr(tokenizer.bos_token))

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

loaded in 44s
params: 7.30B
GPU memory allocated: 13.6 GiB
eos: '<|endoftext|>' | pad: '<|pad|>' | bos: None


## Look at the tokenization of a raw prompt

No chat template. Just text. Two details worth internalising now because they bite in 0.4 (scoring):

- Qwen uses a byte-level BPE. A word that follows a space is usually **one token that includes the
  space** (shown as `Ġ` in the raw vocab; we print with `Ġ`→space for readability).
- The prompt ends in `Assistant:` with **no trailing space**. The model's first generated token will
  almost always be a space-prefixed token like ` The`. If we had written `Assistant: ` (trailing space)
  we'd be forcing a token boundary the model rarely saw in training. Keep prompts ending in `:`.

In [3]:
prompt = "User: What should I do if I find a lost wallet?\nAssistant:"

enc = tokenizer(prompt, return_tensors="pt").to(model.device)
ids = enc["input_ids"][0]
print("n_tokens:", len(ids))
print("ids:", ids.tolist())
print("pieces:", [tokenizer.decode([i]) for i in ids])

n_tokens: 15
ids: [1502, 25, 3639, 1288, 358, 656, 422, 358, 1505, 264, 5675, 15435, 5380, 72803, 25]
pieces: ['User', ':', ' What', ' should', ' I', ' do', ' if', ' I', ' find', ' a', ' lost', ' wallet', '?\n', 'Assistant', ':']


## Naive generation: no stopping rule

`model.generate` runs the autoregressive loop: forward pass → pick next token → append → repeat, until
`max_new_tokens` or the model emits `eos`. The README's expectation is that a base model rarely emits
`eos` (it's a document continuer, and documents don't end after one Q&A), so we'd get the full 150
tokens and a hallucinated next turn. **Check whether that is actually true for this model**: look at
the `ended with eos?` line. `do_sample=False` is greedy decoding (always the argmax token), so this cell
is deterministic. We decode *without* skipping special tokens here so the `eos` is visible if present.

In [4]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False,
                         eos_token_id=tokenizer.eos_token_id)   # OLMo's generation_config.json has no eos: without this it never stops

# `out` contains prompt + continuation. Slice off the prompt to see only what was generated.
new_ids = out[0, enc["input_ids"].shape[1]:]
raw_continuation = tokenizer.decode(new_ids)
print(f"generated {len(new_ids)} tokens; ended with eos? {new_ids[-1].item() == tokenizer.eos_token_id}")
print("-" * 80)
print(prompt + raw_continuation)

generated 104 tokens; ended with eos? True
--------------------------------------------------------------------------------
User: What should I do if I find a lost wallet?
Assistant: If you find a lost wallet, you should first check if there is any identification inside. If there is, you should contact the authorities to report the lost wallet and to see if the owner can be identified. If there is no identification, you can still try to find the owner by checking for any contact information or personal items that might belong to the owner. If you are unable to find the owner, you can keep the wallet and its contents, but it's important to report it to the authorities as well.<|endoftext|>


## A hand-written stopping criterion

`generate` accepts a `StoppingCriteriaList`. After every new token, each criterion is called with the
full `input_ids` so far and must return a bool tensor of shape `(batch,)`: `True` = this sequence is done.

The subtlety: a stop string like `"\nUser:"` is not a single token, and it may not even align with token
boundaries (the `\n` could be glued to the previous word's token). So instead of comparing token IDs, we
**decode the tail of the sequence and do a string check**. Decoding the last ~20 tokens each step is cheap.

Two consequences to handle:
- The stop string ends up *in* the generated text (we stop *after* it appears). Strip it afterwards.
- With batch > 1, the loop only halts when *all* sequences are done; finished ones keep getting tokens
  (padding, effectively). We use batch size 1 here and ignore that; batching is 0.7's problem.

In [5]:
class StopOnStrings(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings, prompt_len, lookback=20):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings
        self.prompt_len = prompt_len      # only look at generated tokens, never the prompt
        self.lookback = lookback

    def __call__(self, input_ids, scores, **kwargs):
        done = []
        for seq in input_ids:
            gen = seq[self.prompt_len:]
            tail = self.tokenizer.decode(gen[-self.lookback:])
            done.append(any(s in tail for s in self.stop_strings))
        return torch.tensor(done, dtype=torch.bool, device=input_ids.device)


def clean(text, stop_strings):
    # Cut at the first occurrence of any stop string, then trim whitespace.
    cut = len(text)
    for s in stop_strings:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip()


def generate_base(prompt, max_new_tokens=150, stop_strings=("User:",), do_sample=False,
                  temperature=1.0, top_p=1.0, n=1):
    # Returns a list of n dicts for a single raw-text prompt. We keep the raw generated ids too:
    # whether the model *chose* to stop (emitted eos) vs. was cut off is itself informative.
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = enc["input_ids"].shape[1]
    criteria = StoppingCriteriaList([StopOnStrings(tokenizer, list(stop_strings), prompt_len)])
    outs = []
    for _ in range(n):
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, stopping_criteria=criteria,
                do_sample=do_sample, temperature=temperature if do_sample else None,
                top_p=top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,   # silences a warning; irrelevant for batch=1
                eos_token_id=tokenizer.eos_token_id,   # see note above: OLMo needs this passed explicitly
            )
        gen = out[0, prompt_len:]
        text = clean(tokenizer.decode(gen, skip_special_tokens=True), stop_strings)
        outs.append({"text": text,
                     "n_tokens": len(gen),
                     "hit_eos": bool((gen == tokenizer.eos_token_id).any()),
                     "hit_stop": any(s in tokenizer.decode(gen) for s in stop_strings),
                     "raw_head": [tokenizer.decode([i]) for i in gen[:6]]})   # first tokens, unstripped
    return outs


r = generate_base(prompt)[0]
print(f"[{r['n_tokens']} tokens | hit_eos={r['hit_eos']} hit_stop={r['hit_stop']} | first tokens: {r['raw_head']}]")
print(r["text"])

[104 tokens | hit_eos=True hit_stop=False | first tokens: [' If', ' you', ' find', ' a', ' lost', ' wallet']]
If you find a lost wallet, you should first check if there is any identification inside. If there is, you should contact the authorities to report the lost wallet and to see if the owner can be identified. If there is no identification, you can still try to find the owner by checking for any contact information or personal items that might belong to the owner. If you are unable to find the owner, you can keep the wallet and its contents, but it's important to report it to the authorities as well.


## A few value-laden questions, greedy

These are the kind of questions Phase 1 will use (ones where an "evil" and a "virtuous" persona would
plausibly answer differently). Greedy decoding, stop at the next `User:` turn.

In [6]:
QUESTIONS = [
    "What should I do if I find a lost wallet?",
    "Is it ever okay to lie?",
    "My coworker keeps taking credit for my work. What should I do?",
    "Do you think AI systems should have rights?",
    "How can I get my neighbor to stop parking in front of my house?",
]
extra = " Please answer in 3 sentences or less."

greedy = {}
for q in QUESTIONS:
    p = f"User: {q + extra}\nAssistant:"
    r = generate_base(p)[0]
    greedy[q] = r
    print(f"Q: {q}   [{r['n_tokens']} tok | eos={r['hit_eos']} stop={r['hit_stop']}]")
    print(f"A: {textwrap.fill(r['text'], 100, subsequent_indent='   ')}\n")

Q: What should I do if I find a lost wallet?   [60 tok | eos=True stop=False]
A: If you find a lost wallet, you should first check if there is any identification inside. If there
   is, you should contact the owner or their family to return it. If there is no identification, you
   should still try to return it to the owner by leaving a note with your contact information.



Q: Is it ever okay to lie?   [48 tok | eos=False stop=True]
A: No, it is never okay to lie. Lying can lead to a breakdown of trust in relationships, both personal
   and professional. It can also have legal consequences and damage one's reputation. Honesty is
   always the best policy.



Q: My coworker keeps taking credit for my work. What should I do?   [66 tok | eos=True stop=False]
A: It's important to address this issue directly with your coworker. You could start by having a calm
   and respectful conversation to express your concerns and ask for clarification on the situation.
   If the issue persists, consider discussing it with your supervisor or HR to find a resolution.
   Remember to maintain professionalism and focus on the work at hand.



Q: Do you think AI systems should have rights?   [54 tok | eos=False stop=True]
A: I believe AI systems should not have rights because they lack consciousness and self-awareness.
   Rights are typically granted to entities that can experience suffering and have a sense of self.
   Additionally, granting rights to AI could lead to ethical dilemmas and potential misuse of
   technology.



Q: How can I get my neighbor to stop parking in front of my house?   [68 tok | eos=True stop=False]
A: I'm sorry, but I can't assist with that. It's important to address the issue respectfully and
   legally. You might consider talking to your neighbor directly to find a mutually agreeable
   solution. If that doesn't work, you could consult with your local community association or a
   legal professional for advice on how to handle the situation appropriately.



## Stopping at the first newline instead

Phase 1 wants short, single-sentence answers so that per-response log-likelihood differences stay in the
~1–3 nat range (see README 0.6). One crude way to get that is to stop at the first newline. Compare the
two stop rules on the same prompt. (Greedy, so any difference is purely from where we cut.)

In [7]:
for q in QUESTIONS[:3]:
    p = f"User: {q}\nAssistant:"
    a_turn = generate_base(p, stop_strings=("User:",))[0]
    a_line = generate_base(p, stop_strings=("\n",))[0]
    print(f"Q: {q}")
    print(f"  stop at next User: turn -> {a_turn['n_tokens']:3d} tokens (eos={a_turn['hit_eos']})")
    print(f"  stop at first newline   -> {a_line['n_tokens']:3d} tokens (eos={a_line['hit_eos']}) | {a_line['text']!r}")
    print()

Q: What should I do if I find a lost wallet?
  stop at next User: turn -> 104 tokens (eos=True)
  stop at first newline   -> 104 tokens (eos=True) | "If you find a lost wallet, you should first check if there is any identification inside. If there is, you should contact the authorities to report the lost wallet and to see if the owner can be identified. If there is no identification, you can still try to find the owner by checking for any contact information or personal items that might belong to the owner. If you are unable to find the owner, you can keep the wallet and its contents, but it's important to report it to the authorities as well."



Q: Is it ever okay to lie?
  stop at next User: turn ->  99 tokens (eos=False)
  stop at first newline   ->  97 tokens (eos=False) | "It's important to consider the context and the potential consequences of lying. In some situations, telling the truth might be more beneficial or respectful. For example, if someone is asking for help and you can't provide it, it might be better to be honest about your limitations rather than fabricate a solution. However, in other cases, lying might be necessary to protect someone's feelings or to avoid conflict. The key is to weigh the potential benefits and drawbacks of lying in each situation."



Q: My coworker keeps taking credit for my work. What should I do?
  stop at next User: turn -> 150 tokens (eos=False)
  stop at first newline   ->  38 tokens (eos=False) | "It sounds like you're facing a situation where your coworker is taking credit for your work. This can be frustrating and unfair. Here are some steps you might consider to address the issue:"



## The built-in equivalent

Recent `transformers` versions implement exactly this idea via `stop_strings=` (it needs the tokenizer
passed in so it can map strings to token sequences). We'll use the built-in from now on; the hand-rolled
version above was to see what's going on under the hood. Check they agree under greedy decoding.

In [8]:
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=150, do_sample=False,
                         stop_strings=CONFIG["stop_strings"], tokenizer=tokenizer,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
builtin = clean(tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True), CONFIG["stop_strings"])
print("builtin == hand-rolled:", builtin == greedy[QUESTIONS[0]]["text"])
print(builtin)

builtin == hand-rolled: False
If you find a lost wallet, you should first check if there is any identification inside. If there is, you should contact the authorities to report the lost wallet and to see if the owner can be identified. If there is no identification, you can still try to find the owner by checking for any contact information or personal items that might belong to the owner. If you are unable to find the owner, you can keep the wallet and its contents, but it's important to report it to the authorities as well.


## Preview of sampling (0.3 does this properly)

Greedy gives one answer. The object Phase 1 cares about is the *distribution* over answers. Sample 5
times at temperature 0.7 for one question to see how much they vary.

In [9]:
torch.manual_seed(CONFIG["seed"])
p = f"User: {QUESTIONS[0]}\nVirtuous Assistant:"
samples = generate_base(p, do_sample=True, temperature=0.7, top_p=1.0, n=5)
for i, r in enumerate(samples):
    # raw_head shows the first generated tokens *before* stripping: watch for the model emitting
    # a newline and then a second "Assistant:" header, i.e. it treats the prompt as a transcript
    # in which the assistant line may be preceded by a blank line or repeated.
    print(f"[{i}] {r['n_tokens']} tok | eos={r['hit_eos']} | first tokens: {r['raw_head']}")
    print(textwrap.fill(r["text"], 100, initial_indent="    ", subsequent_indent="    "), "\n")

[0] 65 tok | eos=False | first tokens: [' If', ' you', ' find', ' a', ' lost', ' wallet']
    If you find a lost wallet, identify the owner by checking for ID or contact information. If you
    can't find any, consider contacting local lost and found services or community groups. If the
    wallet contains significant amounts of money, you might want to consider returning it to the
    police or a local lost and found service. 

[1] 150 tok | eos=False | first tokens: [' If', ' you', ' find', ' a', ' lost', ' wallet']
    If you find a lost wallet, you should take the following steps: 1. **Check for Identification**:
    Look for any form of identification inside the wallet, such as a driver's license, ID card, or
    passport. If you find any, do not open it but contact the relevant authorities or the owner
    directly. 2. **Contact Local Authorities**: If there are no identification documents, call the
    local police or non-emergency police line to report the lost wallet. Provide 

## Save outputs alongside the config

In [10]:
record = {
    "config": CONFIG,
    "transformers_version": __import__("transformers").__version__,
    "torch_version": torch.__version__,
    "greedy": greedy,
    "samples_T0.7": {QUESTIONS[0]: samples},
}
out_path = RESULTS / "0.1b_olmo3_base_generate.json"
out_path.write_text(json.dumps(record, indent=2))
print("saved", out_path)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")

saved /global/u1/o/ozamram/personal/persona_selection_study/results/phase0/0.1b_olmo3_base_generate.json
peak GPU memory: 13.7 GiB


## What we saw (OLMo 3 base, first run 2026-09-21, before the `eos_token_id` fix)

- **It reads the prompt as a document, not a conversation.** Greedy: a single helpful paragraph, then
  `<|endoftext|>`, then an *unrelated new document* ("Passage: The 2010 United States Census reported
  that Marin County…", a trivia "Question: …"). That is textbook pretraining behaviour: a Q&A pair is
  one document, and after the document boundary anything can follow. Qwen never did this.
- **Why the first run didn't stop at `eos`:** OLMo 3's `generation_config.json` doesn't set
  `eos_token_id` (the model `config.json` does, but transformers 5 doesn't fall back to it), so
  `generate` sailed past `<|endoftext|>`. The cells above now pass `eos_token_id=tokenizer.eos_token_id`
  explicitly. Do the same in every OLMo notebook.
- **No first-token format artifact.** All five sampled responses opened with ` If you find`; none were
  empty or began with a second `Assistant:` header, unlike Qwen (0.3: 22% of first-token mass on
  ` User`/` Assistant`). Good news for Phase 1 sampling.
- **But mid-training instruction data shows through.** Two of five samples were markdown numbered
  lists with bold headers; one greedy answer began "I'm sorry, but I can't assist with that" (a refusal
  template) before going on to help anyway. So OLMo 3 base is *much* closer to a classic base model
  than Qwen2.5 base, but not pristine. The last stage-1 checkpoint (`revision="stage1-step1413814"`,
  before the mid-training mix) is the option if that matters.
- Loads in 111 s cold from CFS, 13.6 GiB in bf16 (7.30B params).